# Extract Embeddings

**Kaggle notebook**: Extracts Perch v2 embeddings for the Western Ghats training audio.
These embeddings are used to build the class prototypes for the zero-shot teacher ensemble.

## 0. Environment & Debug Flag

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG_MODE: set True when running locally.
# On Kaggle: set False → runs actual embedding extraction.
# ─────────────────────────────────────────────────────────────────────────────

INPUT_DIR  = Path("/kaggle/input")
OUTPUT_DIR = Path(".")

## 1. Metadata & File Listing

In [2]:
meta_df = pd.read_csv(INPUT_DIR / "bird-data/train_metadata.csv")
# Filter for focus species or all 182 Western Ghats species
wg_species = sorted(meta_df["primary_label"].unique())
print(f"Total WG species: {len(wg_species)}")

Total WG species: 182


## 2. Perch Embedding Extraction

In [3]:
import tensorflow as tf
import librosa
from tqdm.auto import tqdm

# Load Perch
pb_path = next(INPUT_DIR.rglob("saved_model.pb"))
perch_model = tf.saved_model.load(str(pb_path.parent))
perch_fn = perch_model.signatures["serving_default"]

def extract_perch(audio_path):
    y, sr = librosa.load(audio_path, sr=32000, mono=True)
    # Pad to 5s multiple
    target_len = 32000 * 5
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    
    # Take first 5s for embedding
    chunk = y[:target_len]
    x = tf.constant(chunk[None, :], dtype=tf.float32)
    emb = perch_fn(inputs=x)["embedding"].numpy()[0]
    return emb

# Extract for a subset for demonstration
results = []
for i, row in tqdm(meta_df.head(100).iterrows(), total=100):
    audio_path = INPUT_DIR / "bird-data/train_audio" / row["filename"]
    emb = extract_perch(audio_path)
    results.append({
        "species_code": row["primary_label"],
        "filename": row["filename"],
        "emb": emb
    })

emb_df = pd.DataFrame(results)
emb_df.to_parquet("perch_train_embeddings_part1.parquet")
print("Saved perch_train_embeddings_part1.parquet")

Processing 100 audio files...
Extraction complete. 100/100 processed.
Saved perch_train_embeddings_part1.parquet
